# NovelFactory on Kaggle GPU

Run the cells **in order**. Before running anything:

1. **Enable GPU** — right sidebar → *Accelerator* → **GPU T4 x2** (or P100).
2. **Attach model datasets** — *+ Add Data* → *Models* tab:
   - `Qwen/Qwen1.5-4B-Chat` (slug `qwen1-5-4b-chat`)
   - `stabilityai/stable-diffusion-xl-base-1.0` (slug `stable-diffusion-xl-base-1-0`)
3. **Mount the code** — set `REPO_URL` in Cell 2 (public repo), or attach your project zip as a dataset and uncomment the zip block in Cell 2.

Then run: Cell 1 (env) → Cell 2 (mount) → Cell 3 (smoke test) → Cell 4 (planning) → Cell 5 (rendering) → Cell 6 (inspect outputs).

**Download your results** (`sample_project/outputs/*.png`, `final_video.mp4`, manifests) before the session ends — the working directory is wiped when the session stops.

In [ ]:
import subprocess, sys, os

# ── 1a. HuggingFace cache — MUST be a WRITABLE directory ─────────────────────
# /kaggle/input/ is READ-ONLY — setting HF_HOME there causes a crash when
# transformers tries to write its cache index file.
# We use /kaggle/working/hf_cache/ for HF metadata (writable, ~20 GB space).
# The actual model weights are loaded directly from /kaggle/input/ by the
# _find_kaggle_model() helper in local_llm.py using local_files_only=True.
#
# How to attach models (Kaggle sidebar → Data → + Add Data → Models tab):
#   • "Qwen/Qwen1.5-4B-Chat"                    → attaches to /kaggle/input/qwen1-5-4b-chat/
#   • "stabilityai/stable-diffusion-xl-base-1.0" → attaches to /kaggle/input/stable-diffusion-xl-base-1-0/
import os
os.makedirs("/kaggle/working/hf_cache", exist_ok=True)
os.environ["HF_HOME"]               = "/kaggle/working/hf_cache"
os.environ["TRANSFORMERS_CACHE"]    = "/kaggle/working/hf_cache"
os.environ["HUGGINGFACE_HUB_CACHE"] = "/kaggle/working/hf_cache"
os.environ["DIFFUSERS_CACHE"]       = "/kaggle/working/hf_cache"
# Tell local_llm.py where the Kaggle model datasets are mounted:
os.environ["KAGGLE_LLM_INPUT"]      = "/kaggle/input/qwen1-5-4b-chat"
os.environ["KAGGLE_SDXL_INPUT"]     = "/kaggle/input/stable-diffusion-xl-base-1-0"

# ── 1b. Verify GPU ────────────────────────────────────────────────────────────
import torch
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── 1b. Install missing deps (Kaggle already has torch/transformers/diffusers) ─
EXTRA_DEPS = [
    "bitsandbytes>=0.46.1",   # 4-bit quantisation for LLM
    "accelerate",              # device_map="auto" support
    "safetensors",             # fast checkpoint loading
    "sentencepiece",           # Qwen tokenizer
    "python-docx",             # .docx novel input
    "pydantic>=2.0",           # domain models
]

print("\nInstalling extra dependencies...")
for dep in EXTRA_DEPS:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", dep],
        capture_output=True, text=True
    )
    status = "✓" if result.returncode == 0 else "✗ " + result.stderr.strip()
    print(f"  {dep:<40} {status}")

print("\n✅ Environment ready.")

In [ ]:
import os, sys

# ── PREREQUISITE: Attach model datasets BEFORE running this notebook ──────────
# In the Kaggle sidebar: "Data" → "+ Add Data" → search the Models tab
#
#   1. Qwen/Qwen1.5-4B-Chat
#      Kaggle slug: qwen1-5-4b-chat
#      Will appear at: /kaggle/input/qwen1-5-4b-chat/
#
#   2. stabilityai/stable-diffusion-xl-base-1.0
#      Kaggle slug: stable-diffusion-xl-base-1-0
#      Will appear at: /kaggle/input/stable-diffusion-xl-base-1-0/
#
# After attaching, the os.environ lines in Cell 1 point HuggingFace directly
# there — no download occurs, no RAM is consumed during load.
# ─────────────────────────────────────────────────────────────────────────────

# ── Option A: auto-clone from a public repo ───────────────────────────────────
REPO_URL = ""   # ← set to your public repo URL to auto-clone, e.g.
                #   "https://github.com/you/NovelFactory.git"
PROJECT_DIR = "/kaggle/working/NovelFactory"   # ← adjust if needed

# ── Option B: Kaggle Dataset (recommended for private repos) ──────────────────
# 1. Zip your project: zip -r NovelFactory.zip NovelFactory/
# 2. Upload as a Kaggle Dataset named "novel-factory-src"
# 3. Attach it to this notebook, then uncomment:
# import zipfile
# with zipfile.ZipFile("/kaggle/input/novel-factory-src/NovelFactory.zip", "r") as z:
#     z.extractall("/kaggle/working/")

# ── Mount the code ─────────────────────────────────────────────────────────────
if not os.path.isdir(PROJECT_DIR):
    if REPO_URL:
        os.system(f"git clone --depth 1 {REPO_URL} {PROJECT_DIR}")
    if not os.path.isdir(PROJECT_DIR):
        print("Code not found. Mount it first:")
        print("  • Option A: set REPO_URL above to your public repo, or")
        print("  • Option B: attach your uploaded dataset and uncomment the zip block above.")
        raise SystemExit(1)

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

# Add project root to Python path so 'import core.*' works
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
print(f"sys.path[0]      : {sys.path[0]}")

# Quick sanity check
assert os.path.isfile("main.py"), "main.py not found — check PROJECT_DIR"
print("✅ Project mounted.")

In [ ]:
import pkgutil, importlib

print("Running import smoke test across core.*...")
failures = []

for module_info in pkgutil.walk_packages(["core"], prefix="core."):
    try:
        importlib.import_module(module_info.name)
        print(f"  ✓  {module_info.name}")
    except Exception as e:
        failures.append((module_info.name, str(e)))
        print(f"  ✗  {module_info.name}  →  {e}")

if failures:
    print(f"\n❌ {len(failures)} import(s) failed:")
    for name, err in failures:
        print(f"   • {name}: {err}")
    raise RuntimeError("Fix import errors before continuing.")
else:
    print(f"\n✅ All core.* modules imported cleanly.")

In [ ]:
import subprocess, sys
from pathlib import Path

# ── 4a. Write (or specify) the input novel ────────────────────────────────────
NOVEL_PATH = "sample.txt"

NOVEL_TEXT = '''
Chapter 1 — The Forest
Alice stood at the edge of the dark forest, her blue dress catching the moonlight
like a lantern. The trees whispered secrets she couldn't yet understand.

Chapter 2 — The Stranger
A figure emerged from the shadows — tall, cloaked, carrying a lantern of
their own. "You shouldn't be here," he said, "but I'm glad you are."

Chapter 3 — The Choice
She had two paths: the road back to town, safe and familiar, or the winding
trail into the unknown. She had always chosen safety. Tonight felt different.
'''

with open(NOVEL_PATH, "w", encoding="utf-8") as f:
    f.write(NOVEL_TEXT.strip())
print(f"Novel written: {NOVEL_PATH}  ({len(NOVEL_TEXT)} chars)")

# ── 4b. Run planning (StoryBible → SceneSplitter → ShotPlanner →
#                     CastPlanner → CameraPlanner → PromptBuilder) ────────────
# The --novel shorthand auto-creates ./sample_project/ next to the file and
# runs the pipeline there.  Qwen 1.5-4B-Chat loads in 4-bit from the attached
# dataset (see Cell 2) — or downloads to /kaggle/working/hf_cache on first use.
#
# Optional: pin a smaller LLM with --llm-model Qwen/Qwen1.5-0.5B-Chat

result = subprocess.run(
    [sys.executable, "main.py",
     "--novel", NOVEL_PATH,
     "--stage", "plan"],
    text=True
)

if result.returncode != 0:
    raise RuntimeError("Planning pipeline failed — see output above.")
print("\n✅ Planning pipeline complete.")
print("Artifacts:", sorted(Path("sample_project/manifests").glob("*.json")))

In [ ]:
import subprocess, sys
from pathlib import Path

assert Path("sample_project/manifests/prompt_manifest.json").exists(), (
    "prompt_manifest.json not found — run Cell 4 first."
)

result = subprocess.run(
    [sys.executable, "main.py",
     "--novel", "sample.txt",
     "--stage", "render"],
    text=True
)

if result.returncode != 0:
    raise RuntimeError("Diffusion pipeline failed — see output above.")
print("\n✅ Diffusion pipeline complete.")
print("Images:", sorted(Path("sample_project/outputs").glob("*.png")))

In [ ]:
import os, json
from pathlib import Path
from IPython.display import display, Image as IPImage

# The --novel shorthand in Cells 4-5 auto-created ./sample_project/
WORKSPACE = Path("sample_project")

# ── 6a. List all output files ─────────────────────────────────────────────────
print("=== sample_project/ contents ===")
for p in sorted(WORKSPACE.rglob("*")):
    size = f"{p.stat().st_size:>8} B" if p.is_file() else ""
    print(f"  {size}  {p.relative_to(WORKSPACE)}")

# ── 6b. Pretty-print key JSON artifacts (first 40 lines each) ────────────────
for artifact in ["story_bible.json", "scene_manifest.json", "prompt_manifest.json"]:
    path = WORKSPACE / "manifests" / artifact
    if path.exists():
        print(f"\n─── {artifact} ───")
        with open(path, encoding="utf-8") as f:
            lines = f.readlines()
        print("".join(lines[:40]))
        if len(lines) > 40:
            print(f"  ... ({len(lines) - 40} more lines, open file to see all)")

# ── 6d. Display rendered images ───────────────────────────────────────────────
images = sorted(WORKSPACE.rglob("*.png"))
if images:
    print(f"\nRendered {len(images)} image(s):")
    for img_path in images[:12]:
        print(f"  {img_path}")
        display(IPImage(filename=str(img_path), width=512))
else:
    print("\nNo rendered images found yet — run Cell 5 to generate them.")

# ── 6e. VRAM snapshot ─────────────────────────────────────────────────────────
try:
    import torch
    if torch.cuda.is_available():
        alloc   = torch.cuda.memory_allocated() / 1e9
        reservd = torch.cuda.memory_reserved()  / 1e9
        total   = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"\nVRAM: {alloc:.2f} allocated / {reservd:.2f} reserved / {total:.1f} total (GB)")
except ImportError:
    pass